#### MODULES & MAIN DIR

In [2]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import math as math

In [3]:
main_dir = os.getcwd().split('\\')[-1]
if main_dir != 'boston-marathons' and main_dir != 'boston-marathons-Copy1':
    os.chdir("../..")
    main_dir = os.getcwd().split('\\')[-1]
print(main_dir)

boston-marathons-Copy1


#### CLEANING

In [5]:
# raw data import
df_15 = pd.read_csv('data/raw/marathon_results_2015.csv')
df_16 = pd.read_csv('data/raw/marathon_results_2016.csv')
df_17 = pd.read_csv('data/raw/marathon_results_2017.csv')

# new column that represents year of competition
df_15['Year'] = 2015
df_16['Year'] = 2016
df_17['Year'] = 2017

# one big dataset 
dfs = [df_15, df_16, df_17]
df = pd.concat(dfs)

df.head()

,Unnamed: 0,Bib,Name,Age,M/F,City,State,Country,Citizen,Unnamed: 9,...,35K,40K,Pace,Proj Time,Official Time,Overall,Gender,Division,Year,Unnamed: 8
0,0.0,3,"Desisa, Lelisa",25,M,Ambo,NaN,ETH,NaN,NaN,...,1:47:59,2:02:39,0:04:56,-,2:09:17,1,1,1,2015,NaN
1,1.0,4,"Tsegay, Yemane Adhane",30,M,Addis Ababa,NaN,ETH,NaN,NaN,...,1:47:59,2:02:42,0:04:58,-,2:09:48,2,2,2,2015,NaN
2,2.0,8,"Chebet, Wilson",29,M,Marakwet,NaN,KEN,NaN,NaN,...,1:47:59,2:03:01,0:04:59,-,2:10:22,3,3,3,2015,NaN
3,3.0,11,"Kipyego, Bernard",28,M,Eldoret,NaN,KEN,NaN,NaN,...,1:48:03,2:03:47,0:05:00,-,2:10:47,4,4,4,2015,NaN
4,4.0,10,"Korir, Wesley",32,M,Kitale,NaN,KEN,NaN,NaN,...,1:47:59,2:03:27,0:05:00,-,2:10:49,5,5,5,2015,NaN


In [6]:
df.columns

Index(['Unnamed: 0', 'Bib', 'Name', 'Age', 'M/F', 'City', 'State', 'Country',
       'Citizen', 'Unnamed: 9', '5K', '10K', '15K', '20K', 'Half', '25K',
       '30K', '35K', '40K', 'Pace', 'Proj Time', 'Official Time', 'Overall',
       'Gender', 'Division', 'Year', 'Unnamed: 8'],
      dtype='object')

In [7]:
# delete unwanted columns
df = df[['Name', 'M/F', 'Age', 'Country', 'City', 'Pace', '5K', '10K', '15K', '20K', '25K', '30K', '35K', '40K', 'Official Time', 'Year']]
df.head()

,Name,M/F,Age,Country,City,Pace,5K,10K,15K,20K,25K,30K,35K,40K,Official Time,Year
0,"Desisa, Lelisa",M,25,ETH,Ambo,0:04:56,0:14:43,0:29:43,0:44:57,1:00:29,1:16:07,1:32:00,1:47:59,2:02:39,2:09:17,2015
1,"Tsegay, Yemane Adhane",M,30,ETH,Addis Ababa,0:04:58,0:14:43,0:29:43,0:44:58,1:00:28,1:16:07,1:31:59,1:47:59,2:02:42,2:09:48,2015
2,"Chebet, Wilson",M,29,KEN,Marakwet,0:04:59,0:14:43,0:29:43,0:44:57,1:00:29,1:16:07,1:32:00,1:47:59,2:03:01,2:10:22,2015
3,"Kipyego, Bernard",M,28,KEN,Eldoret,0:05:00,0:14:43,0:29:44,0:45:01,1:00:29,1:16:07,1:32:00,1:48:03,2:03:47,2:10:47,2015
4,"Korir, Wesley",M,32,KEN,Kitale,0:05:00,0:14:43,0:29:44,0:44:58,1:00:28,1:16:07,1:32:00,1:47:59,2:03:27,2:10:49,2015


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 79638 entries, 0 to 26409
Data columns (total 16 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Name           79638 non-null  object
 1   M/F            79638 non-null  object
 2   Age            79638 non-null  int64 
 3   Country        79638 non-null  object
 4   City           79637 non-null  object
 5   Pace           79638 non-null  object
 6   5K             79638 non-null  object
 7   10K            79638 non-null  object
 8   15K            79638 non-null  object
 9   20K            79638 non-null  object
 10  25K            79638 non-null  object
 11  30K            79638 non-null  object
 12  35K            79638 non-null  object
 13  40K            79638 non-null  object
 14  Official Time  79638 non-null  object
 15  Year           79638 non-null  int64 
dtypes: int64(2), object(14)
memory usage: 10.3+ MB


In [22]:
# CHANGE COLUMNS TYPE

# list with time value columns
time_columns = ['Pace', '5K', '10K', '15K', '20K', '25K', '30K', '35K', '40K', 'Official Time']

# convert string to timedelta
df[time_columns] = df[time_columns].apply(lambda col: pd.to_timedelta(col, errors='coerce'))

# only 'Pace' column is in minutes - separate that from others and give it float value that represent hours
time_in_hours = [col for col in time_columns if col != 'Pace']
df[time_in_hours] = df[time_in_hours].apply(lambda col: col.dt.total_seconds() / 3600)

# make 'Pace' column represent minutes (min/km)
df['Pace'] = df['Pace'].dt.total_seconds() / 60
df['Pace'] = df['Pace'] / 1.609344

In [10]:
df.head()

,Name,M/F,Age,Country,City,Pace,5K,10K,15K,20K,25K,30K,35K,40K,Official Time,Year
0,"Desisa, Lelisa",M,25,ETH,Ambo,3.065431,0.245278,0.495278,0.749167,1.008056,1.268611,1.533333,1.799722,2.044167,2.154722,2015
1,"Tsegay, Yemane Adhane",M,30,ETH,Addis Ababa,3.086144,0.245278,0.495278,0.749444,1.007778,1.268611,1.533056,1.799722,2.045000,2.163333,2015
2,"Chebet, Wilson",M,29,KEN,Marakwet,3.096500,0.245278,0.495278,0.749167,1.008056,1.268611,1.533333,1.799722,2.050278,2.172778,2015
3,"Kipyego, Bernard",M,28,KEN,Eldoret,3.106856,0.245278,0.495556,0.750278,1.008056,1.268611,1.533333,1.800833,2.063056,2.179722,2015
4,"Korir, Wesley",M,32,KEN,Kitale,3.106856,0.245278,0.495556,0.749444,1.007778,1.268611,1.533333,1.799722,2.057500,2.180278,2015


#### export

In [12]:
df.to_csv('data/clean/boston_clean.csv', index=False)